# 02 - Baseline models and backtest

Compare every registered forecaster on the same rolling-origin backtest that CI
uses, so the numbers here match `reports/metrics.json` exactly.

Executed on every pull request by `pytest --nbmake`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from forecasting.data import (
    DATE_COLUMN,
    aggregate_total,
    load_sales,
    rolling_origin_splits,
)
from forecasting.evaluate import backtest, compare_models
from forecasting.registry import BASELINE_MODEL, available_models

pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (11, 3.5)

HORIZON = 14
N_SPLITS = 5

total = aggregate_total(load_sales())
print(f"series length : {len(total)}")
print(f"models        : {', '.join(available_models())}")
print(f"baseline      : {BASELINE_MODEL}")

## How the backtest splits the series

Expanding window, one origin per fold, oldest first. No test row ever predates
its own training window, which is the whole point.

In [ ]:
splits = list(rolling_origin_splits(total, horizon=HORIZON, n_splits=N_SPLITS))

figure, axis = plt.subplots(figsize=(10, 2.6))
for index, split in enumerate(splits):
    axis.plot(
        [split.train[DATE_COLUMN].min(), split.train[DATE_COLUMN].max()],
        [index, index],
        linewidth=6,
        color="#4c72b0",
        solid_capstyle="butt",
    )
    axis.plot(
        [split.test[DATE_COLUMN].min(), split.test[DATE_COLUMN].max()],
        [index, index],
        linewidth=6,
        color="#c44e52",
        solid_capstyle="butt",
    )

axis.set_yticks(range(len(splits)))
axis.set_yticklabels([f"fold {i}" for i in range(len(splits))])
axis.set_title("Rolling origins: train (blue) and test (red)", loc="left")
figure.tight_layout()

pd.DataFrame(
    {
        "fold": range(len(splits)),
        "train_size": [len(s.train) for s in splits],
        "cutoff": [s.cutoff.date() for s in splits],
        "test_start": [s.test[DATE_COLUMN].min().date() for s in splits],
        "test_end": [s.test[DATE_COLUMN].max().date() for s in splits],
    }
)

## Model comparison

In [ ]:
comparison = compare_models(
    total,
    {"seasonal_naive": {"season_length": 7}, "mean": {"window": 28}, "sarimax": {}},
    horizon=HORIZON,
    n_splits=N_SPLITS,
)
comparison.round(4)

In [ ]:
figure, axis = plt.subplots(figsize=(7, 3.2))
axis.bar(comparison["model"], comparison["mape"] * 100, color="#4c72b0")
for index, value in enumerate(comparison["mape"] * 100):
    axis.text(index, value, f"{value:.2f}%", ha="center", va="bottom", fontsize=9)
axis.set_ylabel("MAPE (%)")
axis.set_title("Backtest MAPE by model (lower is better)", loc="left")
figure.tight_layout()

## Champion vs baseline, fold by fold

An average that hides one catastrophic fold is not an improvement. Check the
spread, not just the pooled number.

In [ ]:
champion_name = comparison.loc[0, "model"]
print(f"champion: {champion_name}")

champion = backtest(
    total, model=champion_name, horizon=HORIZON, n_splits=N_SPLITS, collect_predictions=True
)
baseline = backtest(
    total,
    model=BASELINE_MODEL,
    horizon=HORIZON,
    n_splits=N_SPLITS,
    model_params={"season_length": 7},
)

per_fold = pd.DataFrame(
    {
        "fold": [fold.fold for fold in champion.folds],
        "cutoff": [fold.cutoff for fold in champion.folds],
        f"{champion_name}_mape": [fold.metrics["mape"] for fold in champion.folds],
        f"{BASELINE_MODEL}_mape": [fold.metrics["mape"] for fold in baseline.folds],
    }
)
per_fold["improvement"] = (
    1 - per_fold[f"{champion_name}_mape"] / per_fold[f"{BASELINE_MODEL}_mape"]
)
per_fold.round(4)

In [ ]:
figure, axis = plt.subplots(figsize=(8, 3.2))
width = 0.38
positions = range(len(per_fold))
axis.bar(
    [p - width / 2 for p in positions],
    per_fold[f"{BASELINE_MODEL}_mape"] * 100,
    width,
    label=BASELINE_MODEL,
    color="#8c8c8c",
)
axis.bar(
    [p + width / 2 for p in positions],
    per_fold[f"{champion_name}_mape"] * 100,
    width,
    label=champion_name,
    color="#4c72b0",
)
axis.set_xticks(list(positions))
axis.set_xticklabels(per_fold["cutoff"], rotation=20)
axis.set_ylabel("MAPE (%)")
axis.set_title("Per-fold accuracy: champion vs baseline", loc="left")
axis.legend()
figure.tight_layout()

## Forecasts against actuals

In [ ]:
predictions = champion.predictions
assert predictions is not None

figure, axes = plt.subplots(len(splits), 1, figsize=(9, 2.0 * len(splits)), sharex=False)
for axis, (fold, group) in zip(axes, predictions.groupby("fold"), strict=True):
    axis.plot(
        group[DATE_COLUMN], group["actual"], marker="o", markersize=3, label="actual", color="#333333"
    )
    axis.plot(
        group[DATE_COLUMN],
        group["predicted"],
        marker="o",
        markersize=3,
        label="forecast",
        color="#c44e52",
    )
    axis.set_title(f"fold {fold}", loc="left", fontsize=9)
    axis.tick_params(axis="x", rotation=20)

axes[0].legend(fontsize=8)
figure.tight_layout()

## Residuals

Look for structure. A residual that still cycles weekly means the model has not
captured the seasonality.

In [ ]:
residuals = predictions["actual"] - predictions["predicted"]

figure, (left, right) = plt.subplots(1, 2, figsize=(11, 3.2))
left.scatter(predictions["predicted"], residuals, s=12, alpha=0.6, color="#4c72b0")
left.axhline(0, color="#c44e52", linestyle="--")
left.set_xlabel("forecast")
left.set_ylabel("residual")
left.set_title("Residual vs forecast", loc="left")

right.hist(residuals, bins=25, color="#55a868")
right.axvline(0, color="#c44e52", linestyle="--")
right.set_title("Residual distribution", loc="left")
figure.tight_layout()

print(f"mean residual {residuals.mean():+.2f}, std {residuals.std():.2f}")

In [ ]:
weekday_residual = (
    predictions.assign(weekday=predictions[DATE_COLUMN].dt.day_name())
    .groupby("weekday")[["actual", "predicted"]]
    .mean()
)
weekday_residual["error"] = weekday_residual["predicted"] - weekday_residual["actual"]
weekday_residual.round(2)

## Where this leaves us

`sarimax` clears the seasonal naive baseline, which is what
`eval/thresholds.yml` requires. Known gaps, all of them open work:

- No model uses `on_promotion` or `price`, even though both are known in advance.
- No holiday calendar, so the December lift is absorbed as noise.
- Evaluation is aggregate-only. A model can look fine on `TOTAL` while being
  badly wrong on an individual SKU.

Run the same backtest from the command line with `make backtest`, and check it
against the thresholds with `make gate`.